# Agents and Workflows

Instructor walkthrough notebook for the *Agents and Workflows* module. We'll cover how to handle tasks that don't fit in a single Claude call, the four classic workflow patterns, and where agents start to earn their cost.

## Agenda

1. **Agents and Workflows** — the decision rule + the evaluator-optimizer pattern
2. **Parallelization workflows** — split, fan out, aggregate
3. **Chaining workflows** — let each call focus on one thing
4. **Routing workflows** — classify the request, then specialize
5. **Agents and tools** — flexible plans over abstract tools
6. **Environment inspection** — agents must look at what changed after every action
7. **Workflows vs agents** — when each one wins

Live coding rhythm: read the concept → read the demo description → run the cell → discuss the on-screen output → tweak one knob and re-run.

---
## Setup

Before running anything, make sure:

1. A `.env` file at the repo root contains `ANTHROPIC_API_KEY=sk-ant-...`.
2. `.env` is listed in `.gitignore` — never commit your key, never paste it into a notebook cell.
3. The `anthropic` and `python-dotenv` packages are installed in the active environment.

The next cell installs the dependencies. It's commented out so it only runs when needed.

In [ ]:
# %pip install anthropic python-dotenv

### Demo: load the key and create a client

This cell loads `.env`, instantiates the Anthropic client, picks the workhorse model, and prints a three-line sanity check. If `Key loaded` is `False`, stop and fix `.env` before running anything else.

In [ ]:
import os
import anthropic
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
model = "claude-sonnet-4-6"

print("SDK version:", anthropic.__version__)
print("Model:", model)
print("Key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

### Shared helpers

Three small helpers we'll reuse in every demo: `add_user_message`, `add_assistant_message`, and `chat`. Defining them once keeps later cells short — when you see a demo cell that's only three lines, that's because the helper is doing the request boilerplate.

`chat()` returns just the first text block — fine for the workflow demos. The agent demos later will call `client.messages.create` directly so they can also handle `tool_use` blocks.

In [ ]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})
    return messages

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})
    return messages

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    """Send messages to Claude and return the assistant text."""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system is not None:
        params["system"] = system
    if stop_sequences is not None:
        params["stop_sequences"] = stop_sequences
    response = client.messages.create(**params)
    return response.content[0].text

---
# 1. Agents and Workflows

When a single Claude call can't get the job done — too many constraints, too much branching, multiple artifacts to produce — you reach for one of two strategies: **a workflow** or **an agent**.

### The decision rule

| You know exactly which steps to take | Use a **workflow** |
| --- | --- |
| You can name the goal, but the steps depend on what you find | Use an **agent** |

A workflow is a fixed series of Claude calls — you wrote the loop, you control the order. An agent is a Claude call given tools, where Claude itself decides which tool to call next.

### The evaluator-optimizer pattern

The most common workflow shape is **producer + evaluator + loop**. One call produces an output, a second call grades it, and you re-run the producer with feedback until the grade is good enough.

Image-to-3D-model example from the notes:

```
image  ─▶ describe (Claude)
        ─▶ model with CADQuery (Claude generates Python)
        ─▶ render (Python)
        ─▶ compare to original (Claude — evaluator)
             └─▶ accept  ── done
             └─▶ reject  ── feedback ──▶ back to step 2
```

### Why it matters

Naming the pattern is the easy part — *implementing it is still your job*. The pattern just tells you which loop to write.

### Demo: evaluator-optimizer for a tagline

We ask Claude to write a tagline, then ask Claude (in a fresh conversation) to score it. If the score is below 8 we feed the feedback back into the producer and try again. Watch how the second attempt incorporates the critic's notes verbatim — that's the optimizer half of the loop earning its keep.

Knobs to try live: lower the score threshold to 9 to see the loop work harder, or weaken the producer prompt (drop "single-sentence") to watch the evaluator catch the regression.

In [ ]:
PRODUCT = "a coffee mug that keeps drinks hot for 12 hours"

def write_tagline(product, feedback=None):
    prompt = f"Write a single-sentence tagline for {product}."
    if feedback:
        prompt += f" The previous attempt was rejected with this feedback: {feedback}. Fix those issues."
    return chat(add_user_message([], prompt), temperature=0.9)

def evaluate_tagline(product, tagline):
    prompt = f"""Rate this tagline for {product} on a scale of 1-10 for catchiness AND clarity.
Tagline: {tagline}

Reply in exactly this format:
SCORE: <integer 1-10>
FEEDBACK: <one sentence of specific actionable feedback>"""
    return chat(add_user_message([], prompt), temperature=0.0)

feedback = None
for attempt in range(1, 5):
    tagline = write_tagline(PRODUCT, feedback)
    review = evaluate_tagline(PRODUCT, tagline)
    print(f"--- Attempt {attempt} ---")
    print("Tagline:", tagline)
    print(review, "\n")
    score = int(next(l for l in review.splitlines() if l.startswith("SCORE")).split(":")[1].strip())
    if score >= 8:
        print(f"Accepted at attempt {attempt}.")
        break
    feedback = next(l for l in review.splitlines() if l.startswith("FEEDBACK")).split(":", 1)[1].strip()
else:
    print("Hit max attempts without acceptance.")

> **🏫 During class:**
> 1. Run the cell once and read the first attempt + the evaluator's verdict aloud.
> 2. Say: *"Notice we're not asking one mega-prompt to write **and** judge. We're spending two cheap calls so each one can focus."*
> 3. Ask the room: *"What's the failure mode if the producer and evaluator are the same prompt?"* (Answer: the model rationalizes its own output and never rejects.)

---
# 2. Parallelization Workflows

**Parallelization** = break one complex decision into independent subtasks, run them at the same time, then aggregate.

```
             ┌── evaluate option A ──┐
input  ──▶  ├── evaluate option B ──┤  ──▶  aggregator  ──▶  decision
             └── evaluate option C ──┘
```

### Why split it up

| Benefit | What it buys you |
| --- | --- |
| **Focus** | Each subtask only thinks about one option, not all of them |
| **Modularity** | You can iterate on one sub-prompt without breaking the others |
| **Scalability** | Adding a fifth option is one new call, not a prompt rewrite |
| **Quality** | Reduces the "juggling" failure mode of overstuffed prompts |

Material-selection example from the notes: instead of one prompt asking *"choose between metal/polymer/ceramic/composite for a 250°C kitchen spatula"*, fan out one call per material, then send the four summaries into a final aggregator call.

### Demo: parallel material evaluation, then aggregate

We evaluate four candidate materials, first sequentially and then with a thread pool. The two timings should differ by roughly a factor of N — that's the wall-clock win. The interesting payoff isn't speed, though: it's that each subtask gets to think about exactly one material, so the per-material analysis is sharper. The aggregator at the end is a separate Claude call that only sees the four summaries.

Tweak ideas: add a fifth material to show how it scales, or have the aggregator return JSON for downstream tooling.

In [ ]:
import concurrent.futures
import time

PART = "a kitchen spatula used at 250°C in a hot pan"
MATERIALS = ["silicone", "stainless steel", "hardwood", "nylon"]

def evaluate_material(material):
    prompt = f"""Evaluate {material} as the material for {PART}.
Output:
- 1-sentence verdict
- 2 pros (short bullets)
- 2 cons (short bullets)
- Score 1-10"""
    return material, chat(add_user_message([], prompt), temperature=0.3)

t0 = time.time()
sequential = [evaluate_material(m) for m in MATERIALS]
print(f"Sequential: {time.time() - t0:.1f}s")

t0 = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
    parallel = list(pool.map(evaluate_material, MATERIALS))
print(f"Parallel:   {time.time() - t0:.1f}s\n")

summaries = "\n\n".join(f"### {m}\n{review}" for m, review in parallel)
agg_prompt = f"""Below are independent evaluations of four candidate materials for {PART}.
Pick the single best material and explain in 3 sentences why it beats the runners-up.

{summaries}"""
print("=== Aggregated decision ===\n")
print(chat(add_user_message([], agg_prompt), temperature=0.3))

> **🏫 During class:**
> 1. Run the cell. Point at the `Sequential:` and `Parallel:` lines side by side.
> 2. Say: *"The wall-clock win is real, but the bigger gain is each call only thinks about one material — that's what fixes the 'overstuffed prompt' failure mode."*
> 3. Ask the room: *"What other product decisions decompose this cleanly?"* Steer toward classifications, scoring rubrics, multi-criteria reviews. Then propose: *"What about creative writing — does parallelization fit there?"* (Usually no — you want one coherent voice.)

---
# 3. Chaining Workflows

**Chaining** = a sequence of distinct calls, each focused on one subtask, instead of one mega-prompt.

```
topic  ──▶  search trends  ──▶  pick angle  ──▶  research  ──▶  draft script  ──▶  generate video  ──▶  post
```

### When chaining earns its keep

The signature failure mode it fixes: **constraint-heavy prompts**. You ask Claude for a marketing blurb with five "don'ts" — *don't mention AI, don't use emojis, stay under 80 words, professional tone, no exclamation marks* — and even with repetition, some constraint slips. Asking the model to do everything in one breath is the problem.

Chaining splits that into:

1. **Generate** an imperfect draft.
2. **Audit + rewrite** — a second call whose only job is to find violations and fix them.

Each call has fewer simultaneous constraints, so each call gets them right.

### Demo: generate → audit-and-rewrite

We deliberately stuff the first prompt with constraints ("no AI mentions, no emoji, under 80 words, warm tone"). Often the first draft will trip on at least one. The second call has a single job — find the violations, then output a rewrite that fixes them. Notice the second prompt is much shorter, but more focused.

Tweak ideas: bump the constraint count to 6 or 7 and watch how the single-call version starts losing constraints — then run it through the audit step.

In [ ]:
draft_prompt = (
    "Write a 4-sentence marketing blurb for a meditation app called 'Stillpoint'. "
    "Make it warm and approachable. Don't mention AI, language models, or technology. "
    "Don't use any emojis. Keep it under 80 words."
)
draft = chat(add_user_message([], draft_prompt), temperature=0.8)
print("=== Step 1: draft ===\n")
print(draft)

audit_prompt = f"""Below is a marketing blurb. Audit it for these violations:
- mentions AI / language models / technology
- contains any emojis
- exceeds 80 words

Then output a clean rewrite that fixes every violation while keeping the warm tone.

Blurb:
{draft}

Output exactly two sections:
VIOLATIONS:
<bulleted list, or 'None' if clean>

REWRITE:
<the cleaned-up blurb>"""
print("\n=== Step 2: audit + rewrite ===\n")
print(chat(add_user_message([], audit_prompt), temperature=0.3))

> **🏫 During class:**
> 1. Run the cell once, then re-run a couple of times — chaining shines under variance, so a single run can give a misleading clean result.
> 2. Say: *"The audit prompt is small because it doesn't carry the creative load. Step one creates, step two enforces — neither is doing both."*
> 3. Variation to try live: delete one constraint from `draft_prompt` and add it to `audit_prompt`. Same total constraints across the two calls — does quality stay the same? (Usually yes. The bottleneck is per-call attention, not total information.)

---
# 4. Routing Workflows

**Routing** = first classify the input, then send it down a category-specific pipeline with its own prompt and tools.

```
user input ──▶  classify  ──┬──▶  educational pipeline  (definitions + examples)
                            ├──▶  entertainment pipeline (hook + vivid imagery)
                            └──▶  news pipeline           (headline + facts)
```

### Why route

A single "good" system prompt averages over every type of request. A routed system gives each request the prompt it actually wants. *"How Python decorators work"* deserves a patient teacher; *"the new Dune trailer"* deserves a hyped TikTok host. One general prompt picks one tone and gets the other type wrong every time.

### Demo: classify topic, then route to the matching system prompt

The classifier call uses `temperature=0.0` and a tightly constrained output ("reply with one word") so the routing decision is stable. Then we pick a system prompt from a dictionary and call Claude again with that system. Watch how the same word *"interest rates"* gets a calm broadcaster voice while *"the new Dune trailer"* gets a hype voice — neither prompt knows about the other.

Tweak ideas: add a `comedy` category, or harden the classifier with a `stop_sequences=["\n"]` to defend against paragraph answers.

In [ ]:
PROMPTS = {
    "educational": "Write a 4-sentence script that teaches the topic. Open with a one-line definition, then a concrete example, then why it matters. Tone: patient teacher.",
    "entertainment": "Write a 4-sentence script with a punchy hook in the first line, vivid imagery, and trendy phrasing. Tone: hyped TikTok host.",
    "news": "Write a 4-sentence script that opens with the headline, then 3 supporting facts. Tone: calm, factual broadcaster.",
}

def classify_topic(topic):
    prompt = (
        "Classify this topic into ONE category: educational, entertainment, news. "
        "Reply with the single category word, lowercase, nothing else.\n"
        f"Topic: {topic}"
    )
    return chat(add_user_message([], prompt), temperature=0.0).strip().lower()

def generate_script(topic):
    category = classify_topic(topic)
    print(f"[router] {topic!r} -> {category}")
    return chat(add_user_message([], f"Topic: {topic}"), system=PROMPTS[category], temperature=0.7)

for topic in [
    "how decorators work in Python",
    "the new Dune trailer dropped overnight",
    "interest rates rose 25 basis points this morning",
]:
    print(f"\n--- {topic} ---")
    print(generate_script(topic))

> **🏫 During class:**
> 1. Run the cell so all three categories appear. Read just the first lines of each output side by side — the tonal contrast is the lesson.
> 2. Say: *"The router only knows three words. That's enough — the specialization lives in `PROMPTS`, not in the classifier."*
> 3. Ask the room for a topic that sits between two categories (e.g., *"a viral TikTok teaching SQL joins"*) and run it. Discuss what the classifier picked and whether you'd add a fourth category for that case.

---
# 5. Agents and Tools

**Agents** = a Claude call given tools, where Claude decides which tool to invoke and in what order. Use them when you can name the goal but not the steps.

### The tool-abstraction principle

The strongest agent insight from the notes: **prefer abstract, composable tools over hyper-specialized ones.**

| Specialized toolset | Abstract toolset (what Claude Code uses) |
| --- | --- |
| `refactor_tool`, `install_dependencies`, `run_tests`, `lint_repo` | `bash`, `read_file`, `write_file`, `web_fetch` |
| Each tool only solves one task | Tools combine to solve any task you didn't anticipate |

A small set of flexible tools out-performs a large set of single-purpose ones because the agent gets to combine them in ways the designer never thought of.

Worked example: `get_current_datetime` + `add_duration` + `set_reminder` covers *"remind me in 2 hours"*, *"remind me at 3pm tomorrow"*, and *"when did I last get a reminder"* — three tools, many tasks.

### Demo: a 3-tool reminder agent

We give the agent three abstract tools and one user request: *"set a reminder for 2 hours from now to drink water"*. The agent has to (a) figure out *now*, (b) compute *now + 2h*, and (c) actually set the reminder. Watch the `[tool]` lines in the output — that sequence is the plan Claude generated, not one we hard-coded.

Tweak ideas: change the request to *"every two hours"* and watch the agent fail (we didn't give it a recurrence tool — that's the right behavior) or add a `cancel_reminder` tool and ask it to reschedule.

In [ ]:
from datetime import datetime, timedelta

agent_tools = [
    {
        "name": "get_current_datetime",
        "description": "Returns the current local datetime as an ISO 8601 string.",
        "input_schema": {"type": "object", "properties": {}, "required": []},
    },
    {
        "name": "add_duration",
        "description": "Adds a number of hours to an ISO 8601 datetime string.",
        "input_schema": {
            "type": "object",
            "properties": {
                "iso_datetime": {"type": "string"},
                "hours": {"type": "number"},
            },
            "required": ["iso_datetime", "hours"],
        },
    },
    {
        "name": "set_reminder",
        "description": "Schedules a reminder at a specific ISO datetime with a message.",
        "input_schema": {
            "type": "object",
            "properties": {
                "iso_datetime": {"type": "string"},
                "message": {"type": "string"},
            },
            "required": ["iso_datetime", "message"],
        },
    },
]

def run_agent_tool(name, args):
    if name == "get_current_datetime":
        return datetime.now().isoformat(timespec="seconds")
    if name == "add_duration":
        dt = datetime.fromisoformat(args["iso_datetime"])
        return (dt + timedelta(hours=args["hours"])).isoformat(timespec="seconds")
    if name == "set_reminder":
        return f"OK — reminder scheduled for {args['iso_datetime']}: {args['message']}"
    return f"unknown tool: {name}"

def agent_loop(user_request, tools, run_tool, system=None, max_steps=6):
    messages = [{"role": "user", "content": user_request}]
    for _ in range(max_steps):
        params = {"model": model, "max_tokens": 1000, "tools": tools, "messages": messages}
        if system:
            params["system"] = system
        response = client.messages.create(**params)
        messages.append({"role": "assistant", "content": response.content})
        if response.stop_reason != "tool_use":
            return next((b.text for b in response.content if b.type == "text"), "")
        results = []
        for block in response.content:
            if block.type == "tool_use":
                output = run_tool(block.name, block.input)
                print(f"[tool] {block.name}({block.input}) -> {output}")
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(output)})
        messages.append({"role": "user", "content": results})
    return "(max steps reached)"

print(agent_loop(
    "Set a reminder for 2 hours from now to drink water.",
    tools=agent_tools,
    run_tool=run_agent_tool,
))

> **🏫 During class:**
> 1. Run the cell. Point at the three `[tool]` lines and say: *"Nobody told the agent to call these in this order. That's the plan it picked."*
> 2. Say: *"Three abstract tools just covered a task none of them solve alone — that's the abstraction principle in action."*
> 3. Try a new request live: *"What time is it 90 minutes from now?"* The agent will use just two of the three tools — show how the same toolset handles a different intent.

---
# 6. Environment Inspection

**Environment inspection** = after every tool call, the agent looks at what changed before deciding what to do next.

Why it matters: Claude can't predict the result of an action precisely. A click might open a modal or fail silently. A `write_file` might succeed, get blocked by permissions, or hit the wrong path. If the agent doesn't *look*, it operates blind.

### How real agents inspect

| Agent | What it inspects after every action |
| --- | --- |
| Claude computer use | Screenshots — "did the click do what I expected?" |
| Code-editing agents | Re-read the file after every write |
| Video-generation agents | `ffmpeg` extract frames; Whisper.cpp on the audio |

The pattern: **act → inspect → decide**. Without the inspect step the agent can't detect errors, can't gauge progress, and can't recover from a tool that lied.

### Demo: agent that re-reads after writing

We give the agent a tiny filesystem (just `/tmp/agent_demo`) and tools to list / read / write files. The system prompt tells it to *re-read after writing to verify the change took effect*. Ask it to add a line to `todo.txt`. Watch the trace — you'll see read → write → read again, each tool call interleaved with a model decision.

Tweak ideas: comment out one line of `write_file` (so it silently no-ops) and re-run — the verification re-read is what catches the bug.

In [ ]:
from pathlib import Path

DEMO_DIR = Path("/tmp/agent_demo")
DEMO_DIR.mkdir(exist_ok=True)
(DEMO_DIR / "todo.txt").write_text("- Buy milk\n- Read paper\n")

fs_tools = [
    {
        "name": "list_files",
        "description": "Lists file names in the working directory.",
        "input_schema": {"type": "object", "properties": {}, "required": []},
    },
    {
        "name": "read_file",
        "description": "Returns the full contents of a file by name.",
        "input_schema": {
            "type": "object",
            "properties": {"name": {"type": "string"}},
            "required": ["name"],
        },
    },
    {
        "name": "write_file",
        "description": "Replaces the entire contents of a file. Returns confirmation only — caller is responsible for verifying.",
        "input_schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "content": {"type": "string"},
            },
            "required": ["name", "content"],
        },
    },
]

def run_fs_tool(name, args):
    if name == "list_files":
        return ", ".join(sorted(p.name for p in DEMO_DIR.iterdir()))
    if name == "read_file":
        return (DEMO_DIR / args["name"]).read_text()
    if name == "write_file":
        (DEMO_DIR / args["name"]).write_text(args["content"])
        return f"Wrote {len(args['content'])} chars to {args['name']}."
    return f"unknown tool: {name}"

INSPECT_SYSTEM = (
    "You always read a file before modifying it. After every write, you re-read "
    "the file to verify the change took effect, and you report what you observed."
)

print(agent_loop(
    "Add 'Call mom' as a new bullet line to todo.txt and confirm it was saved.",
    tools=fs_tools,
    run_tool=run_fs_tool,
    system=INSPECT_SYSTEM,
    max_steps=8,
))

> **🏫 During class:**
> 1. Run the cell. Read the trace aloud, naming each phase: *"read — write — read again to verify"*.
> 2. Say: *"The verification read isn't politeness, it's how the agent detects when a tool lied."*
> 3. Variation to try live: temporarily replace the body of `write_file` with `return "Wrote 0 chars."` (skip the actual write). Re-run. The agent should catch the missing line on the second read and try again — that's environment inspection earning its cost.

---
# 7. Workflows vs Agents

Now that we've built both, the comparison.

| Dimension | Workflow | Agent |
| --- | --- | --- |
| **Step sequence** | Predetermined by you | Decided by Claude at runtime |
| **Task focus** | Each call is a small, specific subtask | Each call may juggle planning + acting |
| **Testing** | Deterministic order — easy to test, easy to evaluate | Path varies per run — harder to test |
| **User input** | Requires structured input | Can pull missing input by asking the user |
| **Success rate** | Higher — fewer moving parts | Lower — complexity is delegated to the model |
| **Best when** | You know the steps | You only know the goal |

### The recommendation from the notes

> **Prioritize workflows for reliability. Reach for agents only when flexibility is truly required.**

Users want a 100%-working product over a fancy agent. *Solve the problem reliably first, innovate second.*

### Demo: same task, two strategies

Task: *write a tagline that is provably under 10 words*. We solve it twice — once with a deterministic 2-step workflow, once with an agent that has a `count_words` tool. Both should succeed, but watch the call counts and timings: the workflow runs in two predictable calls; the agent runs in 2-4 calls depending on whether Claude trusts itself or wants to double-check.

Tweak ideas: change the constraint to *"under 6 words"* and re-run — the agent will start retrying, the workflow will need a third step.

In [ ]:
PRODUCT2 = "a backpack made of recycled ocean plastic"

# === Workflow approach: deterministic two-call chain ===
t0 = time.time()
draft = chat(add_user_message([], f"Write a tagline (under 10 words) for: {PRODUCT2}."), temperature=0.7)
audit = chat(
    add_user_message([], f"Count the words in this tagline. If it's under 10, reply 'PASS'. Otherwise reply 'FAIL: <count>'. Tagline: {draft}"),
    temperature=0.0,
)
print(f"--- Workflow ({time.time() - t0:.1f}s) ---")
print("draft:", draft)
print("audit:", audit)

# === Agent approach: tool-using loop, free-form plan ===
count_tools = [{
    "name": "count_words",
    "description": "Returns the number of whitespace-separated words in the input string.",
    "input_schema": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]},
}]

def run_count_tool(name, args):
    if name == "count_words":
        return str(len(args["text"].split()))
    return f"unknown tool: {name}"

t0 = time.time()
result = agent_loop(
    f"Write a tagline for {PRODUCT2}. Use count_words to verify it is strictly under 10 words before answering.",
    tools=count_tools,
    run_tool=run_count_tool,
)
print(f"\n--- Agent ({time.time() - t0:.1f}s) ---")
print("result:", result)

> **🏫 During class:**
> 1. Run the cell. Compare the two timings and the number of `[tool]` lines.
> 2. Say: *"Both worked — the question is which one you'd ship. Which one would you debug at 2am?"* Pause for answers.
> 3. Ask the room: *"What would force us to switch from the workflow to the agent here?"* Steer toward open-ended constraints ("make sure it doesn't rhyme with any competitor's tagline") that the workflow can't pre-bake into a checklist.

---
## Recap and exercises

**Recap**

- Workflows beat one-shot prompts when the task has too many constraints to satisfy in one breath.
- Four canonical shapes: **evaluator-optimizer**, **parallelization**, **chaining**, **routing**.
- Agents trade reliability for flexibility — pick them only when the steps genuinely depend on what's discovered.
- Abstract tools combine; specialized tools don't. Prefer 4 abstract tools over 12 specialized ones.
- Environment inspection — re-reading after acting — is what separates an agent that recovers from a tool failure from one that doesn't.

**Exercises (progressively harder)**

1. **Evaluator-optimizer for code**: write a workflow that asks Claude to write a Python function, then asks Claude to run it (mentally) on three test cases and report any failures. Loop until all three pass.
2. **Parallel summarization**: take three different blog posts (or paragraphs) on the same topic, summarize them in parallel, then aggregate into a single 5-bullet brief.
3. **Constraint-stuffed chaining**: write a one-shot prompt with seven contradictory constraints (no second person, no first person, no questions, exactly 3 sentences, …). Show how a chain of generate-then-audit handles it more reliably than the one-shot.
4. **Routing with overlap**: build a 4-way router (educational / news / entertainment / opinion). Test it on a deliberately ambiguous topic ("why I think Python typing is broken") and decide whether to add a fifth category or improve the classifier prompt.
5. **Inspecting agent**: extend the filesystem agent so it has a `bash` tool. Ask it to find every file containing the word "TODO" and report the count. Watch the inspect-after-act pattern in action.

In [ ]:
# Exercise 1 scaffold — evaluator-optimizer for code.
# Live-build target: write a workflow that loops until all three test cases pass.
#
# def write_function(spec, feedback=None):
#     prompt = f"Write a Python function for: {spec}."
#     if feedback:
#         prompt += f" Previous attempt failed: {feedback}."
#     return chat(add_user_message([], prompt), temperature=0.5)
#
# def evaluate_function(code, test_cases):
#     prompt = f"""Given this Python code:
# {code}
#
# Walk through these test cases mentally and report PASS or FAIL for each:
# {test_cases}
#
# Reply in this format:
# RESULTS:
# - case 1: PASS|FAIL <reason>
# ...
# OVERALL: PASS|FAIL
# """
#     return chat(add_user_message([], prompt), temperature=0.0)
#
# spec = "a function fizzbuzz(n) that returns the standard FizzBuzz string for n"
# tests = "fizzbuzz(3) == 'Fizz'; fizzbuzz(5) == 'Buzz'; fizzbuzz(15) == 'FizzBuzz'"
# feedback = None
# for attempt in range(4):
#     code = write_function(spec, feedback)
#     review = evaluate_function(code, tests)
#     # TODO: parse OVERALL, accept on PASS, otherwise extract feedback and retry